# OMOMO - Layer 2 and Layer 3: compare your run with the paper

`L2_eval.sh ours` runs the paper checkpoint on all 482 test windows and writes one record per window to
`release_data/omomo/L2_outputs/ours/results/metrics_raw.json`. This notebook averages those records with exactly the
same code as `L1_tables.ipynb` and prints, for every column, the paper value, the value recomputed from the paper's
own evaluation (Layer 1) and the value of your run (Layer 2).

Sections whose outputs do not exist yet are skipped.

In [ ]:
import json
import pandas as pd
import omomo_l1 as L

L1 = L.l1_dir()
L2 = L1.parent / 'L2_outputs'
print('Layer-1 folder:', L1)
print('Layer-2 folder:', L2)
paper = pd.Series(L.PAPER_TABLE['WMM (ours)'], index=L.COLUMNS)
l1_pw = L.load_per_window(L1 / L.OURS_DIR)
l1 = pd.Series(L.table_row(l1_pw), index=L.COLUMNS)


def load_run(name):
    d = L2 / name / 'results'
    if not (d / 'metrics_raw.json').exists():
        print(f'pending - {d} does not exist yet')
        return None, None
    pw = L.load_per_window(d)
    extra = json.loads((d / 'metrics.json').read_text())
    return pd.Series(L.table_row(pw), index=L.COLUMNS), {k: extra.get(k) for k in ('R_prec', 'FID')}

## Quick check: `L2_eval.sh ours_smoke`

The smoke row runs the first 9 test windows only (a few minutes). Its averages are compared with the paper's own
evaluation of the same 9 windows.

In [ ]:
smoke_file = L2 / 'ours_smoke' / 'results' / 'metrics_raw.json'
if smoke_file.exists():
    smoke_pw = json.loads(smoke_file.read_text())['per_window']
    stems = {w['stem'] for w in smoke_pw}
    ref_pw = [w for w in l1_pw if w['stem'] in stems]
    assert len(ref_pw) == len(smoke_pw), 'smoke windows not found in the paper evaluation'
    cols = ['MPJPE', 'T_root', 'T_obj', 'O_obj', 'T_e', 'H_feet', 'FS']
    ref, run = L.table_row(ref_pw), L.table_row(smoke_pw)
    print(f'{len(smoke_pw)} windows')
    print(pd.DataFrame({'paper evaluation, same windows': [ref[c] for c in cols],
                        'your smoke run': [run[c] for c in cols]}, index=cols).to_string(float_format=lambda v: f'{v:.3f}'))
else:
    print(f'pending - {smoke_file} does not exist yet')

## Layer 2: the paper checkpoint on all 482 windows

In [ ]:
l2, l2_extra = load_run('ours')
if l2 is not None:
    df = pd.DataFrame({'paper': paper, 'L1 (paper evaluation)': l1, 'L2 (your run)': l2})
    df['L2 - paper'] = df['L2 (your run)'] - df['paper']
    df['L2 - L1 (%)'] = 100 * (df['L2 (your run)'] - df['L1 (paper evaluation)']) / df['L1 (paper evaluation)'].abs()
    print(df.to_string(float_format=lambda v: f'{v:.3f}'))
    rel = df['L2 - L1 (%)'].abs()
    print(f'largest |L2 - L1| = {rel.max():.1f}% ({rel.idxmax()}), median {rel.median():.1f}% over 13 columns')
    print('outside the table:', {k: round(v, 3) for k, v in l2_extra.items() if v is not None})

## Layer 3: your retrained model at step 164,000

`L3_train.sh` retrains the model from scratch and stops at step 164,000, the step of the paper checkpoint;
`L3_probe.sh <gpu> release_data/omomo/L3_outputs/final/last.ckpt 164000` evaluates it on the 482 windows exactly like
Layer 2.

In [ ]:
l3, l3_extra = load_run('probe_step164000')
if l3 is not None:
    df = pd.DataFrame({'paper': paper, 'L1 (paper evaluation)': l1, 'L3 (your retrained model)': l3})
    if l2 is not None:
        df['L2 (paper checkpoint, your run)'] = l2
    df['L3 - L1 (%)'] = 100 * (df['L3 (your retrained model)'] - df['L1 (paper evaluation)']) / df['L1 (paper evaluation)'].abs()
    print(df.to_string(float_format=lambda v: f'{v:.3f}'))
    rel = df['L3 - L1 (%)'].abs()
    print(f'largest |L3 - L1| = {rel.max():.1f}% ({rel.idxmax()}), median {rel.median():.1f}% over 13 columns')
    print('outside the table:', {k: round(v, 3) for k, v in l3_extra.items() if v is not None})